In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, IsolationForest
from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor, TheilSenRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.decomposition import PCA, FastICA
from sklearn.cluster import KMeans
from sklearn.neighbors import LocalOutlierFactor
from scipy import stats
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')

class TargetSpecificOptimizer:
    def __init__(self, train_df, test_df, target_cols, volume_cols):
        self.train_df = train_df
        self.test_df = test_df
        self.target_cols = target_cols
        self.volume_cols = volume_cols
        
        # Target-specific configurations
        self.target_configs = {
            'BlendProperty1': {
                'type': 'volatility',
                'mixing_rule': 'log_linear',
                'robust_scaling': True,
                'outlier_detection': True,
                'feature_selection': 'mutual_info',
                'model_fallback': 'huber',
                'ensemble_weights': [0.3, 0.4, 0.3]  # Conservative blending
            },
            'BlendProperty2': {
                'type': 'density',
                'mixing_rule': 'power',
                'robust_scaling': False,
                'outlier_detection': False,
                'feature_selection': 'f_regression',
                'model_fallback': 'standard',
                'ensemble_weights': [0.4, 0.3, 0.3]
            },
            'BlendProperty3': {
                'type': 'volatility',
                'mixing_rule': 'fractional_power',
                'robust_scaling': True,
                'outlier_detection': True,
                'feature_selection': 'mutual_info',
                'model_fallback': 'theil_sen',
                'ensemble_weights': [0.2, 0.5, 0.3]  # Favor robust methods
            },
            'BlendProperty4': {
                'type': 'density',
                'mixing_rule': 'cubic',
                'robust_scaling': False,
                'outlier_detection': False,
                'feature_selection': 'f_regression',
                'model_fallback': 'standard',
                'ensemble_weights': [0.4, 0.3, 0.3]
            },
            'BlendProperty5': {
                'type': 'temperature',
                'mixing_rule': 'linear',
                'robust_scaling': False,
                'outlier_detection': False,
                'feature_selection': 'f_regression',
                'model_fallback': 'standard',
                'ensemble_weights': [0.5, 0.3, 0.2]  # Trust the base model
            },
            'BlendProperty6': {
                'type': 'density',
                'mixing_rule': 'power',
                'robust_scaling': False,
                'outlier_detection': False,
                'feature_selection': 'f_regression',
                'model_fallback': 'standard',
                'ensemble_weights': [0.4, 0.3, 0.3]
            },
            'BlendProperty7': {
                'type': 'viscosity',
                'mixing_rule': 'log_exponential',
                'robust_scaling': True,
                'outlier_detection': True,
                'feature_selection': 'mutual_info',
                'model_fallback': 'huber',
                'ensemble_weights': [0.2, 0.3, 0.5]  # Heavy robust ensemble
            },
            'BlendProperty8': {
                'type': 'surface_tension',
                'mixing_rule': 'harmonic_weighted',
                'robust_scaling': True,
                'outlier_detection': True,
                'feature_selection': 'mutual_info',
                'model_fallback': 'theil_sen',
                'ensemble_weights': [0.25, 0.45, 0.3]
            },
            'BlendProperty9': {
                'type': 'viscosity',
                'mixing_rule': 'log_exponential',
                'robust_scaling': True,
                'outlier_detection': True,
                'feature_selection': 'mutual_info',
                'model_fallback': 'huber',
                'ensemble_weights': [0.2, 0.3, 0.5]
            },
            'BlendProperty10': {
                'type': 'density',
                'mixing_rule': 'power',
                'robust_scaling': False,
                'outlier_detection': False,
                'feature_selection': 'f_regression',
                'model_fallback': 'standard',
                'ensemble_weights': [0.4, 0.3, 0.3]
            }
        }
    
    def advanced_target_specific_features(self, df, target_property):
        """Generate target-specific features based on chemical domain knowledge"""
        df = df.copy()
        config = self.target_configs[target_property]
        
        # Base mixing rule features
        if config['mixing_rule'] == 'log_linear':
            # For volatile compounds - logarithmic mixing
            for p in range(1, 11):
                df[f'LogLinear_P{p}'] = 0
                for i in range(1, 6):
                    prop_val = np.maximum(df[f'Component{i}_Property{p}'], 1e-10)
                    df[f'LogLinear_P{p}'] += df[f'Component{i}_fraction'] * np.log(prop_val)
                df[f'LogLinear_P{p}'] = np.exp(df[f'LogLinear_P{p}'])
        
        elif config['mixing_rule'] == 'fractional_power':
            # For volatility with fractional powers
            for p in range(1, 11):
                df[f'FracPower_P{p}'] = 0
                for i in range(1, 6):
                    prop_val = np.maximum(df[f'Component{i}_Property{p}'], 1e-10)
                    df[f'FracPower_P{p}'] += df[f'Component{i}_fraction'] * np.power(prop_val, 0.7)
        
        elif config['mixing_rule'] == 'log_exponential':
            # For viscosity - highly nonlinear
            for p in range(1, 11):
                df[f'LogExp_P{p}'] = 0
                for i in range(1, 6):
                    prop_val = np.maximum(df[f'Component{i}_Property{p}'], 1e-10)
                    df[f'LogExp_P{p}'] += df[f'Component{i}_fraction'] * np.log(prop_val)
                df[f'LogExp_P{p}'] = np.exp(df[f'LogExp_P{p}'])
                
                # Add Kendall-Monroe viscosity mixing
                df[f'KM_Viscosity_P{p}'] = 0
                for i in range(1, 6):
                    prop_val = np.maximum(df[f'Component{i}_Property{p}'], 1e-10)
                    df[f'KM_Viscosity_P{p}'] += df[f'Component{i}_fraction'] * np.log(np.log(prop_val + 1))
        
        elif config['mixing_rule'] == 'harmonic_weighted':
            # For surface tension
            for p in range(1, 11):
                df[f'HarmWeighted_P{p}'] = 0
                total_weight = 0
                for i in range(1, 6):
                    weight = df[f'Component{i}_fraction'] * np.sqrt(df[f'Component{i}_Property{p}'])
                    df[f'HarmWeighted_P{p}'] += weight / np.maximum(df[f'Component{i}_Property{p}'], 1e-10)
                    total_weight += weight
                df[f'HarmWeighted_P{p}'] = total_weight / np.maximum(df[f'HarmWeighted_P{p}'], 1e-10)
        
        # Advanced interaction features for problematic targets
        if target_property in ['BlendProperty1', 'BlendProperty3', 'BlendProperty7', 'BlendProperty8', 'BlendProperty9']:
            # Component clustering embeddings
            component_features = [f'Component{i}_Property{j}' for i in range(1, 6) for j in range(1, 11)]
            component_matrix = df[component_features].values.reshape(-1, 5, 10)
            
            # Intra-component similarity
            for i in range(1, 6):
                component_props = df[[f'Component{i}_Property{j}' for j in range(1, 11)]].values
                df[f'Component{i}_prop_variance'] = np.var(component_props, axis=1)
                df[f'Component{i}_prop_skew'] = stats.skew(component_props, axis=1)
                df[f'Component{i}_prop_kurt'] = stats.kurtosis(component_props, axis=1)
            
            # Inter-component correlations
            for i in range(1, 5):
                for j in range(i+1, 6):
                    corr_features = []
                    for p in range(1, 11):
                        corr = np.corrcoef(df[f'Component{i}_Property{p}'], df[f'Component{j}_Property{p}'])[0, 1]
                        corr_features.append(corr if not np.isnan(corr) else 0)
                    df[f'Component{i}_{j}_correlation'] = np.mean(corr_features)
            
            # Component dominance patterns
            for i in range(1, 6):
                df[f'Component{i}_dominance_score'] = 0
                for p in range(1, 11):
                    max_prop = df[[f'Component{k}_Property{p}' for k in range(1, 6)]].max(axis=1)
                    df[f'Component{i}_dominance_score'] += (df[f'Component{i}_Property{p}'] == max_prop).astype(int)
            
            # Property-specific nonlinear transformations
            for p in range(1, 11):
                linear_mix = sum(df[f'Component{i}_Property{p}'] * df[f'Component{i}_fraction'] for i in range(1, 6))
                df[f'Property{p}_excess'] = df[f'Component1_Property{p}'] - linear_mix  # Excess property
                df[f'Property{p}_ideal_deviation'] = np.abs(df[f'Property{p}_excess'])
        
        return df
    
    def detect_and_handle_outliers(self, X_train, y_train, target_property):
        """Advanced outlier detection and handling"""
        config = self.target_configs[target_property]
        
        if not config['outlier_detection']:
            return X_train, y_train, np.ones(len(X_train), dtype=bool)
        
        # Multi-method outlier detection
        outlier_methods = []
        
        # 1. Isolation Forest on features
        iso_forest = IsolationForest(contamination=0.1, random_state=42)
        feature_outliers = iso_forest.fit_predict(X_train) == -1
        outlier_methods.append(feature_outliers)
        
        # 2. Local Outlier Factor
        lof = LocalOutlierFactor(contamination=0.1)
        lof_outliers = lof.fit_predict(X_train) == -1
        outlier_methods.append(lof_outliers)
        
        # 3. Target-based outlier detection (Z-score)
        z_scores = np.abs(stats.zscore(y_train))
        target_outliers = z_scores > 3
        outlier_methods.append(target_outliers)
        
        # 4. Residual-based outlier detection (using simple linear model)
        from sklearn.linear_model import LinearRegression
        lr = LinearRegression()
        lr.fit(X_train, y_train)
        residuals = y_train - lr.predict(X_train)
        residual_outliers = np.abs(stats.zscore(residuals)) > 2.5
        outlier_methods.append(residual_outliers)
        
        # Combine outlier detection methods (at least 2 methods agree)
        outlier_votes = np.sum(outlier_methods, axis=0)
        final_outliers = outlier_votes >= 2
        
        print(f"  🔍 Outlier detection for {target_property}: {np.sum(final_outliers)} outliers found")
        
        # Create sample weights instead of removing outliers
        sample_weights = np.ones(len(X_train))
        sample_weights[final_outliers] = 0.3  # Downweight outliers
        
        return X_train, y_train, sample_weights
    
    def target_specific_feature_selection(self, X_train, y_train, target_property, k=300):
        """Target-specific feature selection"""
        config = self.target_configs[target_property]
        
        if config['feature_selection'] == 'mutual_info':
            selector = SelectKBest(score_func=mutual_info_regression, k=k)
        else:
            selector = SelectKBest(score_func=f_regression, k=k)
        
        X_train_selected = selector.fit_transform(X_train, y_train)
        return X_train_selected, selector
    
    def create_target_specific_ensemble(self, X_train, y_train, target_property, sample_weights):
        """Create target-specific ensemble with fallback strategies"""
        config = self.target_configs[target_property]
        
        # Base models with target-specific configurations
        models = {}
        
        # LightGBM with target-specific params
        if target_property in ['BlendProperty1', 'BlendProperty3', 'BlendProperty7', 'BlendProperty8', 'BlendProperty9']:
            lgb_params = {
                'objective': 'regression',
                'metric': 'mape',
                'boosting_type': 'gbdt',
                'num_leaves': 31,
                'learning_rate': 0.03,  # Lower for stability
                'feature_fraction': 0.8,
                'bagging_fraction': 0.8,
                'bagging_freq': 5,
                'min_child_samples': 20,
                'reg_alpha': 0.3,  # Higher regularization
                'reg_lambda': 0.3,
                'n_estimators': 2000,
                'verbose': -1
            }
        else:
            lgb_params = {
                'objective': 'regression',
                'metric': 'mape',
                'boosting_type': 'gbdt',
                'num_leaves': 63,
                'learning_rate': 0.05,
                'feature_fraction': 0.9,
                'bagging_fraction': 0.9,
                'bagging_freq': 3,
                'min_child_samples': 10,
                'reg_alpha': 0.1,
                'reg_lambda': 0.1,
                'n_estimators': 1500,
                'verbose': -1
            }
        
        models['lgb'] = lgb.LGBMRegressor(**lgb_params)
        
        # XGBoost with target-specific params
        if target_property in ['BlendProperty1', 'BlendProperty3', 'BlendProperty7', 'BlendProperty8', 'BlendProperty9']:
            xgb_params = {
                'objective': 'reg:squarederror',
                'n_estimators': 1500,
                'learning_rate': 0.03,
                'max_depth': 6,
                'min_child_weight': 3,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'reg_alpha': 0.3,
                'reg_lambda': 0.3,
                'random_state': 42
            }
        else:
            xgb_params = {
                'objective': 'reg:squarederror',
                'n_estimators': 1000,
                'learning_rate': 0.05,
                'max_depth': 8,
                'min_child_weight': 1,
                'subsample': 0.9,
                'colsample_bytree': 0.9,
                'reg_alpha': 0.1,
                'reg_lambda': 0.1,
                'random_state': 42
            }
        
        models['xgb'] = xgb.XGBRegressor(**xgb_params)
        
        # Robust fallback models for problematic targets
        if config['model_fallback'] == 'huber':
            models['fallback'] = HuberRegressor(epsilon=1.35, alpha=0.01)
        elif config['model_fallback'] == 'theil_sen':
            models['fallback'] = TheilSenRegressor(random_state=42)
        else:
            models['fallback'] = Ridge(alpha=1.0, random_state=42)
        
        return models
    
    def uncertainty_aware_ensemble(self, predictions, target_property):
        """Uncertainty-aware ensemble weighting"""
        config = self.target_configs[target_property]
        weights = np.array(config['ensemble_weights'])
        
        # Adjust weights based on prediction uncertainty
        pred_std = np.std(predictions, axis=0)
        uncertainty_factor = 1.0 - (pred_std / np.max(pred_std))
        
        # For high-uncertainty predictions, favor robust models
        if target_property in ['BlendProperty1', 'BlendProperty3', 'BlendProperty7', 'BlendProperty8', 'BlendProperty9']:
            # Increase weight of fallback model for uncertain predictions
            high_uncertainty_mask = uncertainty_factor < 0.5
            if np.any(high_uncertainty_mask):
                weights = np.array([0.2, 0.2, 0.6])  # Favor fallback
        
        return weights
    
    def train_target_specific_model(self, target_property):
        """Train target-specific optimized model"""
        print(f"\n🎯 Training optimized model for {target_property}...")
        
        # Generate target-specific features
        train_features = self.advanced_target_specific_features(self.train_df, target_property)
        test_features = self.advanced_target_specific_features(self.test_df, target_property)
        
        # Prepare data
        y_train = train_features[target_property]
        feature_cols = [col for col in train_features.columns if col not in self.target_cols + ['ID']]
        
        X_train = train_features[feature_cols]
        X_test = test_features[feature_cols]
        
        # Handle infinite values
        X_train = X_train.replace([np.inf, -np.inf], np.nan)
        X_test = X_test.replace([np.inf, -np.inf], np.nan)
        
        # Fill NaN values
        X_train = X_train.fillna(X_train.median())
        X_test = X_test.fillna(X_train.median())
        
        # Outlier detection and handling
        X_train, y_train, sample_weights = self.detect_and_handle_outliers(X_train, y_train, target_property)
        
        # Feature selection
        X_train_selected, feature_selector = self.target_specific_feature_selection(X_train, y_train, target_property)
        X_test_selected = feature_selector.transform(X_test)
        
        # Target-specific scaling
        config = self.target_configs[target_property]
        if config['robust_scaling']:
            scaler = RobustScaler()
        else:
            scaler = PowerTransformer(method='yeo-johnson')
        
        X_train_scaled = scaler.fit_transform(X_train_selected)
        X_test_scaled = scaler.transform(X_test_selected)
        
        # Create models
        models = self.create_target_specific_ensemble(X_train_scaled, y_train, target_property, sample_weights)
        
        # Cross-validation with sample weights
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        oof_predictions = np.zeros((len(X_train_scaled), len(models)))
        test_predictions = np.zeros((len(X_test_scaled), len(models)))
        
        for model_idx, (model_name, model) in enumerate(models.items()):
            print(f"    📊 Training {model_name}...")
            
            model_test_preds = []
            model_oof_preds = np.zeros(len(X_train_scaled))
            
            for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_scaled)):
                X_train_fold = X_train_scaled[train_idx]
                X_val_fold = X_train_scaled[val_idx]
                y_train_fold = y_train.iloc[train_idx]
                y_val_fold = y_train.iloc[val_idx]
                weights_fold = sample_weights[train_idx]
                
                if 'lgb' in model_name:
                    model.fit(
                        X_train_fold, y_train_fold,
                        sample_weight=weights_fold,
                        eval_set=[(X_val_fold, y_val_fold)],
                        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
                    )
                elif 'xgb' in model_name:
                    model.fit(
                        X_train_fold, y_train_fold,
                        sample_weight=weights_fold
                    )
                else:
                    model.fit(X_train_fold, y_train_fold, sample_weight=weights_fold)
                
                val_pred = model.predict(X_val_fold)
                test_pred = model.predict(X_test_scaled)
                
                model_oof_preds[val_idx] = val_pred
                model_test_preds.append(test_pred)
            
            oof_predictions[:, model_idx] = model_oof_preds
            test_predictions[:, model_idx] = np.mean(model_test_preds, axis=0)
        
        # Uncertainty-aware ensemble
        weights = self.uncertainty_aware_ensemble(test_predictions, target_property)
        
        # Final predictions
        final_oof = np.average(oof_predictions, axis=1, weights=weights)
        final_test = np.average(test_predictions, axis=1, weights=weights)
        
        # Advanced post-processing
        final_test = self.advanced_postprocessing(final_test, y_train, target_property)
        
        # Calculate score
        score = mean_absolute_percentage_error(y_train, final_oof)
        print(f"    🎯 {target_property} Optimized MAPE: {score:.4f}")
        
        return final_test, score
    
    def advanced_postprocessing(self, predictions, y_train, target_property):
        """Advanced post-processing for specific targets"""
        config = self.target_configs[target_property]
        
        # Robust clipping based on training data
        q1, q99 = np.percentile(y_train, [1, 99])
        iqr = np.percentile(y_train, 75) - np.percentile(y_train, 25)
        
        # More aggressive clipping for problematic targets
        if target_property in ['BlendProperty1', 'BlendProperty3', 'BlendProperty7', 'BlendProperty8', 'BlendProperty9']:
            lower_bound = q1 - 0.5 * iqr
            upper_bound = q99 + 0.5 * iqr
        else:
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q99 + 1.5 * iqr
        
        predictions = np.clip(predictions, lower_bound, upper_bound)
        
        # Smoothing for high-variance targets
        if target_property in ['BlendProperty7', 'BlendProperty9']:
            # Apply median filter for smoothing
            from scipy.ndimage import median_filter
            predictions = median_filter(predictions, size=3)
        
        # Bias correction
        train_mean = np.mean(y_train)
        pred_mean = np.mean(predictions)
        bias = train_mean - pred_mean
        
        # Apply bias correction with dampening
        predictions = predictions + 0.5 * bias
        
        return predictions


# Main execution
def run_target_specific_optimization():
    """Run the complete target-specific optimization pipeline"""
    
    # Load data (assuming train.csv and test.csv exist)
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')
    
    target_cols = [f'BlendProperty{i}' for i in range(1, 11)]
    volume_cols = [f'Component{i}_fraction' for i in range(1, 6)]
    
    # Initialize optimizer
    optimizer = TargetSpecificOptimizer(train, test, target_cols, volume_cols)
    
    # Current MAPE scores for comparison
    current_mape = {
        'BlendProperty1': 1.2367,
        'BlendProperty2': 0.5022,
        'BlendProperty3': 1.0066,
        'BlendProperty4': 0.7236,
        'BlendProperty5': 0.2339,
        'BlendProperty6': 0.5660,
        'BlendProperty7': 2.4120,
        'BlendProperty8': 0.9871,
        'BlendProperty9': 1.4616,
        'BlendProperty10': 0.7089
    }
    
    print("🚀 Starting Target-Specific Optimization Pipeline...")
    print("=" * 70)
    
    # Train optimized models for each target
    final_predictions = {}
    optimized_scores = {}
    
    for target in target_cols:
        predictions, score = optimizer.train_target_specific_model(target)
        final_predictions[target] = predictions
        optimized_scores[target] = score
        
        # Compare with current performance
        improvement = current_mape[target] - score
        improvement_pct = (improvement / current_mape[target]) * 100
        
        print(f"    📈 Improvement: {improvement:.4f} ({improvement_pct:.1f}%)")
    
    # Create final submission
    submission = pd.DataFrame()
    submission['ID'] = range(1, len(final_predictions[target_cols[0]]) + 1)
    
    for target in target_cols:
        submission[target] = final_predictions[target]
    
    # Final results
    print("\n" + "=" * 70)
    print("📊 TARGET-SPECIFIC OPTIMIZATION RESULTS")
    print("=" * 70)
    
    avg_current = np.mean(list(current_mape.values()))
    avg_optimized = np.mean(list(optimized_scores.values()))
    overall_improvement = avg_current - avg_optimized
    
    print(f"\n🎯 Overall Performance:")
    print(f"   Current Average MAPE: {avg_current:.4f}")
    print(f"   Optimized Average MAPE: {avg_optimized:.4f}")
    print(f"   Overall Improvement: {overall_improvement:.4f} ({(overall_improvement/avg_current)*100:.1f}%)")
    
    print(f"\n📋 Target-by-Target Results:")
    for target in target_cols:
        current = current_mape[target]
        optimized = optimized_scores[target]
        improvement = current - optimized
        print(f"   {target}: {current:.4f} → {optimized:.4f} ({improvement:.4f})")
    
    # Save submission
    submission.to_csv('submission_target_optimized.csv', index=False)
    print(f"\n✅ Optimized submission saved as 'submission_target_optimized.csv'")
    
    # Performance prediction
    expected_score = 100 * (1 - avg_optimized)
    print(f"🎲 Expected Competition Score: {expected_score:.1f}%")
    
    if avg_optimized < 0.5:
        print("🎉 TARGET ACHIEVED! Average MAPE < 0.5!")
    else:
        print(f"🎯 Target: < 0.5, Current: {avg_optimized:.4f}")
    
    return submission, optimized_scores



KeyboardInterrupt: 